# [EXPERIMENT_NAME] Pretraining of Swin Transformer Encoder

[DESCRIPTION: one paragraph describing the pretraining recipe, what
makes it different from other experiments, and what hypothesis it tests.]

**Notebook structure:**
1. Configuration
2. Imports & device
3. Data loading
4. Model definition ← **recipe-specific**
5. Loss & augmentation ← **recipe-specific**
6. Overfit sanity check
7. Full training loop
8. Post-training diagnostics
9. Save

## 1. Configuration

All parameters live here. Change these cells, nothing else.

In [ ]:
from types import SimpleNamespace


In [ ]:
cfg = SimpleNamespace(
    seed=42,
    output_root="./outputs",
    tag="experiment_v1",              # short tag for this run
)


In [ ]:
data_cfg = SimpleNamespace(
    data_root="../../data/patches_128",
    exclude_patterns=["KONTROLA"],
    val_split=0.1,
    batch_size=32,
    num_workers=2,
    pin_memory=True,
)


In [ ]:
model_cfg = SimpleNamespace(
    in_channels=3,
    spatial_dims=2,
    img_size=128,
    feature_size=48,
    patch_size=2,
    window_size=7,
    dropout_path_rate=0.0,
    use_checkpoint=False,
    # --- Recipe-specific model params go here ---
    # e.g. decoder_type, mask_token_dim, projection_dim, etc.
)


In [ ]:
import torch.nn as nn

overfit_cfg = SimpleNamespace(
    lr=5e-4,
    weight_decay=0.05,
    grad_clip_norm=5.0,
    max_steps=2000,
    log_every=100,
    patch_indices=[3, 4, 50, 1000],
    mask_ratio=0.6,
    mask_block_size=16,
)


In [ ]:
train_cfg = SimpleNamespace(
    epochs=200,
    lr=1.5e-4,
    weight_decay=0.05,
    warmup_epochs=10,
    grad_clip_norm=5.0,
    experiment_name="TEMPLATE_pretrain",   # CHANGE THIS
    encoder_save_name="pretrained_encoder.pt",
)


## 2. Imports & Device

In [ ]:
import os, sys, time, math, csv, json, copy
from datetime import datetime

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, Subset, random_split
from tqdm.auto import tqdm
from monai.networks.nets.swin_unetr import SwinTransformer

sys.path.insert(0, os.path.abspath("../.."))
from data_utils.patch_dataset import PatchDataset


In [ ]:
n_cpus = int(os.environ.get("SLURM_CPUS_PER_TASK",
             os.environ.get("PBS_NUM_PPN", 4)))
os.environ.setdefault("OPENBLAS_NUM_THREADS", str(max(1, n_cpus // 2)))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
generator = torch.Generator().manual_seed(cfg.seed)
torch.manual_seed(cfg.seed)

if torch.cuda.is_available():
    print(f"{torch.cuda.get_device_name(0)}, "
          f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("Running on CPU")


## 3. Data Loading

In [ ]:
dataset = PatchDataset(
    root=data_cfg.data_root,
    exclude_patterns=data_cfg.exclude_patterns,
)
print(f"Total patches: {len(dataset)}")

sample = dataset[0]
print(f"Sample shape: {sample.shape}, dtype: {sample.dtype}")
print(f"Value range:  [{sample.min():.4f}, {sample.max():.4f}]")
assert sample.ndim == 3
assert 0 <= sample.min() and sample.max() <= 1.0 + 1e-6


In [ ]:
n_val = int(len(dataset) * data_cfg.val_split)
n_train = len(dataset) - n_val
train_dataset, val_dataset = random_split(
    dataset, [n_train, n_val], generator=generator
)
print(f"Train: {n_train}, Validation: {n_val}")

train_loader = DataLoader(
    train_dataset, batch_size=data_cfg.batch_size, shuffle=True,
    num_workers=data_cfg.num_workers, pin_memory=data_cfg.pin_memory,
    persistent_workers=data_cfg.num_workers > 0,
    prefetch_factor=2 if data_cfg.num_workers > 0 else None,
)
val_loader = DataLoader(
    val_dataset, batch_size=data_cfg.batch_size, shuffle=False,
    num_workers=data_cfg.num_workers, pin_memory=data_cfg.pin_memory,
    persistent_workers=data_cfg.num_workers > 0,
    prefetch_factor=2 if data_cfg.num_workers > 0 else None,
)
print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")

data_cfg.n_val = n_val
data_cfg.n_train = n_train


## 4. Model Definition

**TODO (recipe-specific):** Define the pretraining model class here.
Must expose:
- `model.swinViT` — the MONAI SwinTransformer encoder (saved after training)
- `model.encode(x)` — returns deepest feature map `(B, enc_ch, S, S)`
- `model.encode_pooled(x)` — returns `(B, enc_ch)` for collapse diagnostics
- `model.forward(x, mask)` — full forward pass, returns reconstruction/prediction

In [ ]:
# ┌──────────────────────────────────────────────────────────────────────┐
# │  TODO: PASTE YOUR MODEL CLASS HERE                                  │
# │                                                                     │
# │  Requirements:                                                      │
# │    - __init__(self, args)  takes model_cfg                          │
# │    - self.swinViT          MONAI SwinTransformer                    │
# │    - encode(x)             returns deepest feature map              │
# │    - encode_pooled(x)      returns (B, enc_ch) global descriptor   │
# │    - forward(x, mask)      returns reconstruction                  │
# └──────────────────────────────────────────────────────────────────────┘

# class MyPretrainer(nn.Module):
#     ...

raise NotImplementedError("Define your model class above")


## 5. Loss & Augmentation

**TODO (recipe-specific):** Define loss function(s) and any augmentation
(e.g. masking, cutout, contrastive views).

In [ ]:
# ┌──────────────────────────────────────────────────────────────────────┐
# │  TODO: DEFINE LOSS FUNCTION(S)                                     │
# │                                                                     │
# │  Must accept (pred, target, mask) or similar.                      │
# │  Return a scalar tensor.                                           │
# └──────────────────────────────────────────────────────────────────────┘

# def compute_loss(model_output, x, mask):
#     ...
#     return loss

# ⚠️  AGENT NOTE: When filling this in for a specific recipe,
# also update the OVERFIT LOOP (section 6) and the TRAINING LOOP
# (section 7) to use the actual loss + forward pass, not this
# placeholder. If the recipe uses teacher-student (e.g. iBOT, DINO),
# the overfit loop must call the real teacher/student forward, not
# just model(x, mask). The training loop forward pass must also match.

raise NotImplementedError("Define your loss function above")


In [ ]:
# Masking function (shared across recipes that use masking).
# Skip or replace if your recipe uses a different augmentation.
def random_block_mask(img, block_size, mask_ratio):
    """(B, 1, H, W) float mask, 1 = masked. SimMIM-style block masking."""
    B, _, H, W = img.shape
    assert H % block_size == 0 and W % block_size == 0
    gh, gw = H // block_size, W // block_size
    n_blocks = gh * gw
    n_mask = int(math.ceil(n_blocks * mask_ratio))
    noise = torch.rand(B, n_blocks, device=img.device)
    rank = noise.argsort(dim=1)
    flat = (rank < n_mask).float()
    mask = flat.view(B, 1, gh, gw)
    mask = F.interpolate(mask, scale_factor=block_size, mode="nearest")
    return mask


## 6. Overfit Sanity Check

Train on 4 fixed patches with a fixed mask. If loss does not drop to
near zero, something is wrong with the model or loss, not the data.

In [ ]:
# Build overfit mini-batch
x_fixed = torch.stack(
    [dataset[i] for i in overfit_cfg.patch_indices]
).to(device)
print(f"Overfit tensor: {x_fixed.shape}")

fig, axes = plt.subplots(1, len(overfit_cfg.patch_indices),
                         figsize=(4 * len(overfit_cfg.patch_indices), 4))
for i, ax in enumerate(axes):
    img = x_fixed[i].cpu()
    ax.imshow(img.permute(1, 2, 0).numpy() if img.shape[0] >= 3
              else img[0].numpy(), cmap="gray")
    ax.set_title(f"idx={overfit_cfg.patch_indices[i]}")
    ax.axis("off")
plt.suptitle("Overfit set")
plt.tight_layout()
plt.show()


In [ ]:
model = MyPretrainer(model_cfg).to(device)   # TODO: replace MyPretrainer

n_total = sum(p.numel() for p in model.parameters())
n_enc = sum(p.numel() for p in model.swinViT.parameters())
print(f"Total: {n_total:,}  Encoder: {n_enc:,}  Decoder: {n_total - n_enc:,}")


In [ ]:
# Fixed mask for deterministic overfit
torch.manual_seed(cfg.seed)
fixed_mask = random_block_mask(
    x_fixed, overfit_cfg.mask_block_size, overfit_cfg.mask_ratio
).to(device)
print(f"Mask: {fixed_mask.shape}, ratio: {fixed_mask.mean():.2f}")


In [ ]:
# ── Overfit loop ──
model.train()
opt_test = torch.optim.AdamW(
    model.parameters(), lr=overfit_cfg.lr,
    weight_decay=overfit_cfg.weight_decay,
)
scaler_test = torch.amp.GradScaler(device.type, enabled=device.type == "cuda")
loss_history = []

for step in tqdm(range(overfit_cfg.max_steps), desc="Overfit"):
    opt_test.zero_grad(set_to_none=True)
    with torch.amp.autocast(device.type, enabled=device.type == "cuda"):
        recon = model(x_fixed, fixed_mask)
        loss = compute_loss(recon, x_fixed, fixed_mask)  # TODO: match your loss
    # ⚠️  AGENT NOTE: If your recipe uses teacher-student (iBOT, DINO),
    # replace model(x_fixed, fixed_mask) + compute_loss above with the
    # actual student_forward/teacher_forward + recipe loss. The overfit
    # must test the REAL training pipeline, not a substitute.
    scaler_test.scale(loss).backward()
    scaler_test.unscale_(opt_test)
    torch.nn.utils.clip_grad_norm_(model.parameters(), overfit_cfg.grad_clip_norm)
    scaler_test.step(opt_test)
    scaler_test.update()
    loss_history.append(loss.item())
    if step % overfit_cfg.log_every == 0:
        tqdm.write(f"Step {step:4d} | loss: {loss.item():.6f}")

print(f"Final loss: {loss_history[-1]:.6f}")


In [ ]:
# Overfit loss curve
plt.figure(figsize=(10, 4))
plt.plot(loss_history)
plt.xlabel("Step"); plt.ylabel("Loss")
plt.title("Overfit loss curve"); plt.yscale("log")
plt.grid(True, alpha=0.3); plt.show()


In [ ]:
# Visual reconstruction of overfit patches
model.eval()
with torch.no_grad():
    recon = model(x_fixed, fixed_mask)

n_p = len(overfit_cfg.patch_indices)
fig, axes = plt.subplots(3, n_p, figsize=(4 * n_p, 12))
for i in range(n_p):
    orig   = x_fixed[i].cpu().permute(1, 2, 0).numpy()
    masked = (x_fixed[i] * (1 - fixed_mask[i])).cpu().permute(1, 2, 0).numpy()
    rec    = recon[i].cpu().permute(1, 2, 0).numpy()
    axes[0, i].imshow(orig);   axes[0, i].set_title(f"original");    axes[0, i].axis("off")
    axes[1, i].imshow(masked); axes[1, i].set_title("masked");       axes[1, i].axis("off")
    axes[2, i].imshow(rec);    axes[2, i].set_title("reconstruction"); axes[2, i].axis("off")
plt.suptitle("Overfit reconstructions")
plt.tight_layout(); plt.show()


## 7. Full Training

Re-initialize model and train on the full dataset.
Only run after overfit check passes.

In [ ]:
def make_lr_lambda(warmup_epochs, total_epochs):
    """Linear warmup then cosine decay to 0."""
    def lr_lambda(epoch):
        if epoch < warmup_epochs:
            return epoch / max(1, warmup_epochs)
        progress = (epoch - warmup_epochs) / max(1, total_epochs - warmup_epochs)
        return 0.5 * (1.0 + math.cos(math.pi * progress))
    return lr_lambda


### Training setup

In [ ]:
run_timestamp = datetime.now().strftime("%Y_%m_%d_%H%M%S")
save_dir = os.path.join(
    cfg.output_root,
    f"{train_cfg.experiment_name}_{run_timestamp}"
)
os.makedirs(save_dir, exist_ok=True)
print(f"Output directory: {save_dir}")

model = MyPretrainer(model_cfg).to(device)   # TODO: replace MyPretrainer
optimizer = torch.optim.AdamW(
    model.parameters(), lr=train_cfg.lr,
    weight_decay=train_cfg.weight_decay, betas=(0.9, 0.999),
)
scheduler = torch.optim.lr_scheduler.LambdaLR(
    optimizer, make_lr_lambda(train_cfg.warmup_epochs, train_cfg.epochs),
)
scaler = torch.amp.GradScaler(device.type, enabled=device.type == "cuda")

n_total = sum(p.numel() for p in model.parameters())
n_enc = sum(p.numel() for p in model.swinViT.parameters())
print(f"Model re-initialized. Total: {n_total:,}  Encoder: {n_enc:,}")


### Experiment metadata + CSV logger

In [ ]:
experiment_meta = {
    "experiment_name":   train_cfg.experiment_name,
    "tag":               cfg.tag,
    "start_time":        datetime.now().isoformat(),
    "device":            str(device),
    "gpu_name":          torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
    "pytorch_version":   torch.__version__,
    "n_train":           len(train_loader.dataset),
    "n_val":             len(val_loader.dataset),
    "img_size":          model_cfg.img_size,
    "in_channels":       model_cfg.in_channels,
    "feature_size":      model_cfg.feature_size,
    "total_params":      n_total,
    "encoder_params":    n_enc,
    "optimizer":         "AdamW",
    "lr":                train_cfg.lr,
    "weight_decay":      train_cfg.weight_decay,
    "batch_size":        data_cfg.batch_size,
    "epochs":            train_cfg.epochs,
    "warmup_epochs":     train_cfg.warmup_epochs,
    "scheduler":         "linear warmup + cosine decay",
    "mixed_precision":   device.type == "cuda",
    "grad_clip_norm":    train_cfg.grad_clip_norm,
    "seed":              cfg.seed,
    # TODO: add recipe-specific metadata (mask_ratio, loss name, etc.)
}

print("=" * 60)
for k, v in experiment_meta.items():
    print(f"  {k:25s}: {v}")
print("=" * 60)

csv_path = os.path.join(save_dir, f"{train_cfg.experiment_name}_log.csv")
csv_fields = [
    "epoch", "train_loss", "val_loss", "lr",
    "epoch_time_s", "train_time_s", "val_time_s",
    "best_val_loss", "best_epoch",
    "grad_norm_mean", "grad_norm_max",
]
csv_file = open(csv_path, "w", newline="")
csv_writer = csv.DictWriter(csv_file, fieldnames=csv_fields)
csv_writer.writeheader()
print(f"Logging to: {csv_path}")


### (Optional) Resume from checkpoint

Skip for a fresh run. Set `resume_path` to continue from a saved checkpoint.

In [ ]:
import csv as _csv

resume_path = None  # e.g. "./outputs/EXPERIMENT_.../best_model.pt"

start_epoch = 1
_resumed_history = None

if resume_path is not None:
    ckpt = torch.load(resume_path, map_location=device)
    model.load_state_dict(ckpt["model_state_dict"])
    optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    scheduler.load_state_dict(ckpt["scheduler_state_dict"])
    scaler.load_state_dict(ckpt["scaler_state_dict"])
    start_epoch = ckpt["epoch"] + 1
    best_val_loss = ckpt["val_loss"]
    best_epoch = ckpt["epoch"]
    print(f"Resumed from epoch {ckpt['epoch']}, val_loss={ckpt['val_loss']:.6f}")

    # Load history from CSV in the same directory
    ckpt_dir = os.path.dirname(resume_path)
    csv_candidates = [f for f in os.listdir(ckpt_dir) if f.endswith("_log.csv")]
    if csv_candidates:
        csv_resume_path = os.path.join(ckpt_dir, csv_candidates[0])
        with open(csv_resume_path, "r") as f:
            _resumed_history = [r for r in _csv.DictReader(f)
                                if int(r["epoch"]) < start_epoch]
        print(f"Loaded {len(_resumed_history)} epochs of history")
else:
    print("Fresh training run")


### Training loop

In [ ]:
if resume_path is None:
    best_val_loss = float("inf")
    best_epoch = 0
    start_epoch = 1

if _resumed_history:
    train_losses = [float(r["train_loss"]) for r in _resumed_history]
    val_losses   = [float(r["val_loss"])   for r in _resumed_history]
    lr_history   = [float(r["lr"])         for r in _resumed_history]
    grad_norms   = [float(r["grad_norm_mean"]) for r in _resumed_history]
    epoch_times  = [float(r["epoch_time_s"])   for r in _resumed_history]
    print(f"Pre-filled {len(train_losses)} epochs of history")
else:
    train_losses = []
    val_losses   = []
    lr_history   = []
    grad_norms   = []
    epoch_times  = []

total_train_start = time.time()


In [ ]:
for epoch in range(start_epoch, train_cfg.epochs + 1):
    # ── Train ──
    model.train()
    running_loss = 0.0
    epoch_grad_norms = []
    train_start = time.time()

    pbar = tqdm(train_loader,
                desc=f"Epoch {epoch}/{train_cfg.epochs} [train]", leave=False)
    for batch_idx, x in enumerate(pbar, 1):
        x = x.to(device, non_blocking=True)
        mask = random_block_mask(x, overfit_cfg.mask_block_size, overfit_cfg.mask_ratio)

        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast(device.type, enabled=device.type == "cuda"):
            recon = model(x, mask)
            loss = compute_loss(recon, x, mask)  # TODO: match your loss
    # ⚠️  AGENT NOTE: Replace forward pass + loss above with your recipe's
    # actual forward (e.g. student_forward + teacher_forward for iBOT).
    # Must match the overfit loop. Also add any per-step updates
    # (e.g. EMA teacher update, centering update).

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        total_norm = torch.nn.utils.clip_grad_norm_(
            model.parameters(), max_norm=train_cfg.grad_clip_norm)
        epoch_grad_norms.append(total_norm.item())
        scaler.step(optimizer)
        scaler.update()
        running_loss += loss.item()
        pbar.set_postfix(loss=f"{running_loss / batch_idx:.6f}")

    train_loss = running_loss / len(train_loader)
    train_losses.append(train_loss)
    train_time = time.time() - train_start
    scheduler.step()

    # ── Validate ──
    model.eval()
    val_running = 0.0
    val_start = time.time()
    with torch.no_grad():
        for x in tqdm(val_loader,
                      desc=f"Epoch {epoch}/{train_cfg.epochs} [val]", leave=False):
            x = x.to(device, non_blocking=True)
            mask = random_block_mask(x, overfit_cfg.mask_block_size, overfit_cfg.mask_ratio)
            with torch.amp.autocast(device.type, enabled=device.type == "cuda"):
                recon = model(x, mask)
                loss = compute_loss(recon, x, mask)
    # ⚠️  AGENT NOTE: Val forward must match train forward exactly.
            val_running += loss.item()
    val_loss = val_running / len(val_loader)
    val_losses.append(val_loss)
    val_time = time.time() - val_start

    # ── Log ──
    current_lr = scheduler.get_last_lr()[0]
    lr_history.append(current_lr)
    epoch_time = train_time + val_time
    epoch_times.append(epoch_time)
    mean_grad = sum(epoch_grad_norms) / len(epoch_grad_norms)
    max_grad = max(epoch_grad_norms)
    grad_norms.append(mean_grad)

    improved = ""
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_epoch = epoch
        improved = " *best*"
        torch.save({
            "epoch":               epoch,
            "model_state_dict":    model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "scaler_state_dict":   scaler.state_dict(),
            "val_loss":            val_loss,
            "train_loss":          train_loss,
            "model_cfg":           vars(model_cfg),
            "train_cfg":           {k: v for k, v in vars(train_cfg).items()
                                    if isinstance(v, (int, float, str, bool))},
        }, os.path.join(save_dir, "best_model.pt"))

    print(f"Epoch {epoch:3d}/{train_cfg.epochs} | "
          f"Train: {train_loss:.6f} | Val: {val_loss:.6f} | "
          f"LR: {current_lr:.2e} | Time: {epoch_time:.1f}s | "
          f"GradNorm: {mean_grad:.3f}{improved}")

    csv_writer.writerow({
        "epoch": epoch, "train_loss": train_loss, "val_loss": val_loss,
        "lr": current_lr, "epoch_time_s": epoch_time,
        "train_time_s": train_time, "val_time_s": val_time,
        "best_val_loss": best_val_loss, "best_epoch": best_epoch,
        "grad_norm_mean": mean_grad, "grad_norm_max": max_grad,
    })
    csv_file.flush()

total_train_time = time.time() - total_train_start
csv_file.close()
print(f"\nTotal training time: {total_train_time / 60:.1f} min")


## 8. Post-Training Diagnostics

### Reconstruction on 4 validation patches

In [ ]:
model.eval()
x_show = next(iter(val_loader))[:4].to(device)
mask_show = random_block_mask(x_show, overfit_cfg.mask_block_size, overfit_cfg.mask_ratio)

with torch.no_grad():
    recon_show = model(x_show, mask_show)

fig, axes = plt.subplots(3, 4, figsize=(16, 12))
for i in range(4):
    orig   = x_show[i].cpu().permute(1, 2, 0).numpy()
    masked = (x_show[i] * (1 - mask_show[i])).cpu().permute(1, 2, 0).numpy()
    rec    = recon_show[i].cpu().permute(1, 2, 0).numpy()
    axes[0, i].imshow(orig);   axes[0, i].set_title(f"Val {i}: original"); axes[0, i].axis("off")
    axes[1, i].imshow(masked); axes[1, i].set_title("masked");             axes[1, i].axis("off")
    axes[2, i].imshow(rec);    axes[2, i].set_title("reconstruction");     axes[2, i].axis("off")
plt.suptitle("Validation reconstructions")
plt.tight_layout()
fig.savefig(os.path.join(save_dir, "val_reconstructions.png"), dpi=200, bbox_inches="tight")
plt.show()


### Dimensional collapse (SVD of pooled features)

In [ ]:
N_DIAG = min(2048, len(dataset))
diag_loader = DataLoader(
    Subset(dataset, range(N_DIAG)),
    batch_size=data_cfg.batch_size, shuffle=False, num_workers=0,
)

model.eval()
latents = []
with torch.no_grad():
    for x in tqdm(diag_loader, desc="Encoding for collapse check"):
        x = x.to(device, non_blocking=True)
        latents.append(model.encode_pooled(x).cpu())

Z = torch.cat(latents, dim=0)
D_lat = Z.shape[1]
Z = Z - Z.mean(dim=0, keepdim=True)
_, S, _ = torch.svd(Z)
S_norm = S / S[0]

s2 = (S ** 2) / (S ** 2).sum()
log_s2 = torch.log(s2 + 1e-12)
effective_rank = torch.exp(-(s2 * log_s2).sum()).item()

print(f"Pooled latent: {Z.shape}, effective rank: {effective_rank:.1f} / {D_lat} "
      f"({effective_rank / D_lat * 100:.1f}%)")
print(f"Top-10 SVs: {[round(v, 4) for v in S_norm[:10].tolist()]}")

if effective_rank < D_lat * 0.1:
    print("WARNING: LIKELY DIMENSIONAL COLLAPSE")
elif effective_rank < D_lat * 0.3:
    print("CAUTION: Moderate collapse")
else:
    print("OK: No significant collapse")


In [ ]:
# Save collapse diagnostics
np.save(os.path.join(save_dir, "sv_spectrum.npy"), S_norm.numpy())
collapse_diag = {
    "n_patches": int(Z.shape[0]),
    "latent_dim": D_lat,
    "effective_rank": effective_rank,
    "top10_sv": [round(v, 6) for v in S_norm[:10].tolist()],
}
with open(os.path.join(save_dir, "collapse_diag.json"), "w") as f:
    json.dump(collapse_diag, f, indent=2)

# SV spectrum plot
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(S_norm.numpy(), lw=1.5)
axes[0].set_xlabel("Index"); axes[0].set_ylabel("s_i / s_0")
axes[0].set_title("SV Spectrum (linear)"); axes[0].grid(True, alpha=0.3)
axes[1].semilogy(S_norm.numpy(), lw=1.5)
axes[1].set_xlabel("Index"); axes[1].set_ylabel("s_i / s_0 (log)")
axes[1].set_title("SV Spectrum (log)")
axes[1].axhline(y=0.01, color="red", ls="--", alpha=0.5, label="1% of max")
axes[1].legend(); axes[1].grid(True, alpha=0.3)
plt.suptitle(f"Effective rank: {effective_rank:.1f} / {D_lat}")
plt.tight_layout()
fig.savefig(os.path.join(save_dir, "sv_spectrum.png"), dpi=200, bbox_inches="tight")
plt.show()


## 9. Save

### Save experiment summary

In [ ]:
experiment_meta["end_time"]         = datetime.now().isoformat()
experiment_meta["total_time_hours"] = round(total_train_time / 3600, 3)
experiment_meta["best_val_loss"]    = best_val_loss
experiment_meta["best_epoch"]       = best_epoch
experiment_meta["final_train_loss"] = train_losses[-1]
experiment_meta["final_val_loss"]   = val_losses[-1]
experiment_meta["effective_rank"]   = effective_rank

meta_path = os.path.join(save_dir, f"{train_cfg.experiment_name}_meta.json")
with open(meta_path, "w") as f:
    json.dump(experiment_meta, f, indent=2)
print(f"Saved metadata: {meta_path}")


### Loss curves

In [ ]:
epochs_range = range(1, len(train_losses) + 1)
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(epochs_range, train_losses, label="Train")
axes[0].plot(epochs_range, val_losses,   label="Val")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss")
axes[0].set_title("Loss (linear)"); axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].semilogy(epochs_range, train_losses, label="Train")
axes[1].semilogy(epochs_range, val_losses,   label="Val")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Loss (log)")
axes[1].set_title("Loss (log)"); axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.suptitle(f"{train_cfg.experiment_name}")
plt.tight_layout()
fig.savefig(os.path.join(save_dir, "loss_curves.png"), dpi=200, bbox_inches="tight")
fig.savefig(os.path.join(save_dir, "loss_curves.pdf"), bbox_inches="tight")
plt.show()


### Save encoder weights

Load best checkpoint and extract just the SwinViT encoder state dict.
This is what gets loaded into `SwinUNETR.swinViT` for fine-tuning.

In [ ]:
best_ckpt = torch.load(
    os.path.join(save_dir, "best_model.pt"), map_location=device,
)
model.load_state_dict(best_ckpt["model_state_dict"])

encoder_path = os.path.join(save_dir, train_cfg.encoder_save_name)
torch.save(model.swinViT.state_dict(), encoder_path)
print(f"Saved encoder: {encoder_path}")
print(f"Best epoch: {best_ckpt['epoch']}, val_loss: {best_ckpt['val_loss']:.6f}")
